# Patient-Adaptive Interaction-Term Composite

This notebook runs the Patient-Adaptive interaction model separately from the SRM Global Linear notebook. The model allows MRI feature weights to vary with participant demographic/genetic modulators, while clinical scores remain benchmarks only and are not used for training or tuning.

**Validation rule:** split by `subject`, not by `pair_id`, so `V1V2` and `V2V3` intervals from the same participant are never separated across train/test.


In [7]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary
from src.reporting.model_performance import assemble_performance_rows, save_one_performance_csv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"
split_group_col = "subject"
RANDOM_SEED = DEFAULT_CONFIG.random_state
N_BOOT = 300
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} pair-interval visit rows, {len(imaging_cols)} imaging features")
print("Available participant columns:", [c for c in ["age", "sex", "gender", "gaa_1", "gaa_2", "onset_age", "disease_duration", "site"] if c in long_df.columns])


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_pairs_drop3poms.csv: 414 pair-interval visit rows, 146 imaging features
Available participant columns: ['age', 'sex', 'gaa_1', 'gaa_2', 'onset_age', 'disease_duration', 'site']


## 1. Experiment Settings

The Patient-Adaptive model uses all available demographic/genetic modulators in the long-form modelling table. `site` is intentionally not included because it is an acquisition/scanner/site variable, not a participant demographic feature.


In [8]:
selection_method = "none"
selection_k = int(globals().get("selection_k", 8))
ADAPTIVE_CV_N_SPLITS = globals().get("ADAPTIVE_CV_N_SPLITS", DEFAULT_CONFIG.cv_n_splits)
adaptive_z_clip_grid = [None, 4.0, 2.75]
optimization_rows = []

demographic_modulators = [
    c for c in ["age", "sex", "gender", "gaa_1", "gaa_2", "onset_age", "disease_duration"]
    if c in long_df.columns
]
if not demographic_modulators:
    raise ValueError("No demographic/genetic modulators found in the long-form modelling data.")

print({
    "selection_method": selection_method,
    "selection_k": selection_k,
    "adaptive_cv_n_splits": ADAPTIVE_CV_N_SPLITS,
    "adaptive_z_clip_grid": adaptive_z_clip_grid,
    "demographic_modulators": demographic_modulators,
})


{'selection_method': 'none', 'selection_k': 8, 'adaptive_cv_n_splits': 5, 'adaptive_z_clip_grid': [None, 4.0, 2.75], 'demographic_modulators': ['age', 'sex', 'gaa_1', 'gaa_2', 'onset_age', 'disease_duration']}


## 2. Patient-Adaptive Tuning

Each candidate is evaluated with participant-grouped folds. The displayed tuning evidence keeps `d12` and `d23` separate and uses mean annual validation `d_z = (d12 + d23) / 2` for performance review. For this exploratory Patient-Adaptive comparison, the final candidate is the numerically best all-demographic-modulator configuration.


In [9]:
adaptive_trials = []

for adaptive_z_clip in adaptive_z_clip_grid:
    adaptive_config = Config(
        interaction_en_alpha_grid=(0.01, 0.03, 0.1, 0.3, 0.7),
        interaction_en_l1_ratio_grid=(0.2, 0.5, 0.8, 1.0),
        interaction_inner_cv_splits=5,
        interaction_z_clip=adaptive_z_clip,
    )
    start = time.time()
    res = interaction_loocv(
        long_df,
        imaging_cols,
        demographic_modulators,
        subject_col=subject_col,
        visit_col="visit",
        selection_method=selection_method,
        k=selection_k,
        cv_n_splits=ADAPTIVE_CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
        config=adaptive_config,
        compute_ci=False,
    )
    interval_summary = adjacent_pair_interval_effect_summary(
        res["oof_df"],
        pair_col=subject_col,
        visit_col="visit",
        score_col="score",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    res = {**res, **annual_tuning_diagnostics(interval_summary)}
    row = optimization_row(
        model="Patient-Adaptive",
        params={
            "selection_method": selection_method,
            "modulators": ",".join(demographic_modulators),
            "z_clip": adaptive_z_clip,
            "regularization": "ElasticNet",
            "alpha_grid": "0.01|0.03|0.1|0.3|0.7",
            "l1_ratio_grid": "0.2|0.5|0.8|1.0",
            "trial_id": len(adaptive_trials),
        },
        result=res,
        runtime_sec=time.time() - start,
        notes="patient-adaptive model using all available demographic/genetic modulators; final choice uses numerically best mean annual d_z from V1->V2 and V2->V3",
    )
    adaptive_trials.append((res, row, adaptive_config, interval_summary))
    optimization_rows.append(row)

adaptive_optimization_df = optimization_log([row for _, row, _, _ in adaptive_trials], sort_by="mean_validation_annual_dz")
print("Patient-Adaptive tuning candidates evaluated:", len(adaptive_optimization_df))

adaptive_review = tuning_recommendation(adaptive_optimization_df)
print("Numerically best patient-adaptive configuration")
display(pd.DataFrame([adaptive_review["raw_best"]]))
print("One-SE / near-optimal patient-adaptive candidate count:", len(adaptive_review["near_optimal"]))
print("Recommended patient-adaptive configuration by implemented hierarchy")
display(pd.DataFrame([adaptive_review["recommended"]]))
print(adaptive_review["summary"])
print("Human-verification summary")
display(tuning_verification_summary(adaptive_review))


Patient-Adaptive tuning candidates evaluated: 3
Numerically best patient-adaptive configuration


,param_selection_method,param_modulators,param_z_clip,param_regularization,param_alpha_grid,param_l1_ratio_grid,param_trial_id,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,none,"age,sex,gaa_1,gaa_2,onset_age,disease_duration",4.0,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0,1,0.679098,0.439552,0.559325,0.239545,0.71816,NaN,NaN,NaN,NaN,NaN,1.0


One-SE / near-optimal patient-adaptive candidate count: 1
Recommended patient-adaptive configuration by implemented hierarchy


,param_selection_method,param_modulators,param_z_clip,param_regularization,param_alpha_grid,param_l1_ratio_grid,param_trial_id,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank,directional_consistency,score_ranking_stability
0,none,"age,sex,gaa_1,gaa_2,onset_age,disease_duration",4.0,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0,1,0.679098,0.439552,0.559325,0.239545,0.71816,inf,-inf,-inf,-inf,NaN,1.0,-inf,-inf


Recommended candidate is also the raw best by mean annual validation d_z; the consistency, progression-probability, simplicity, and stability tie-breakers did not select a different row.
Human-verification summary


,item,value
0,Best raw-performance parameters,"{'selection_method': 'none', 'modulators': 'ag..."
1,Recommended parameters,"{'selection_method': 'none', 'modulators': 'ag..."
2,Difference in performance,0.0
3,Reason for recommendation,Recommended candidate is also the raw best by ...
4,Any instability/warning,No automatic warning.


## 3. Final Patient-Adaptive OOF Evaluation

The chosen configuration is rerun with confidence intervals enabled. These are genuine out-of-fold scores from grouped participant splits.


In [10]:
adaptive_choice = adaptive_review["raw_best"]
print("Patient-Adaptive final choice: numerically best all-demographic-modulator configuration")
display(pd.DataFrame([adaptive_choice]))

chosen_idx = int(adaptive_choice.get("param_trial_id", adaptive_choice.name))
adaptive_res, adaptive_best_row, adaptive_config, adaptive_interval_summary = adaptive_trials[chosen_idx]
best_adaptive_z_clip = adaptive_best_row.get("param_z_clip", np.nan)
best_adaptive_z_clip = None if pd.isna(best_adaptive_z_clip) else float(best_adaptive_z_clip)
adaptive_config.interaction_z_clip = best_adaptive_z_clip

adaptive_res = interaction_loocv(
    long_df,
    imaging_cols,
    demographic_modulators,
    subject_col=subject_col,
    visit_col="visit",
    selection_method=selection_method,
    k=selection_k,
    cv_n_splits=ADAPTIVE_CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    config=adaptive_config,
    compute_ci=True,
)
adaptive_intervals = adjacent_pair_interval_effect_summary(
    adaptive_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
adaptive_res = {**adaptive_res, **annual_tuning_diagnostics(adaptive_intervals)}

print("Final annual interval diagnostics")
display(adaptive_intervals)

final_summary = pd.DataFrame([{
    "model": "Patient-Adaptive",
    "best_modulators": ", ".join(demographic_modulators),
    "best_z_clip": best_adaptive_z_clip,
    "dz_v1_v2": adaptive_res["dz_v1_v2"],
    "dz_v2_v3": adaptive_res["dz_v2_v3"],
    "mean_annual_d_z": adaptive_res["mean_validation_annual_dz"],
    "annual_interval_gap": adaptive_res["annual_interval_gap"],
    "p_progression": adaptive_res["p_progression"],
    "pooled_pair_d_z_reference": adaptive_res["d_score"],
    "ci_low": adaptive_res["d_ci_low"],
    "ci_high": adaptive_res["d_ci_high"],
    "n_subject_pairs": adaptive_res["n_subjects"],
    "cv_mode": adaptive_res.get("cv_mode"),
    "split_group_col": adaptive_res.get("split_group_col"),
}])
display(final_summary)


Patient-Adaptive final choice: numerically best all-demographic-modulator configuration


,param_selection_method,param_modulators,param_z_clip,param_regularization,param_alpha_grid,param_l1_ratio_grid,param_trial_id,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,none,"age,sex,gaa_1,gaa_2,onset_age,disease_duration",4.0,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0,1,0.679098,0.439552,0.559325,0.239545,0.71816,NaN,NaN,NaN,NaN,NaN,1.0


Final annual interval diagnostics


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,106,0.305987,0.450579,0.679098,0.496567,0.880980,0.811321
1,V2->V3,96,0.236508,0.538066,0.439552,0.257320,0.657084,0.625000


,model,best_modulators,best_z_clip,dz_v1_v2,dz_v2_v3,mean_annual_d_z,annual_interval_gap,p_progression,pooled_pair_d_z_reference,ci_low,ci_high,n_subject_pairs,cv_mode,split_group_col
0,Patient-Adaptive,"age, sex, gaa_1, gaa_2, onset_age, disease_dur...",4.0,0.679098,0.439552,0.559325,0.239545,0.71816,0.552491,0.410609,0.698604,202,group_kfold,subject


## 4. Compare With Current Best Model

This table compares the selected Patient-Adaptive model with the current best SRM Global Linear model from `srm_composite.ipynb`. The comparison uses the same annual interval metrics from the optimization logs.


In [11]:
srm_log_path = REPO_ROOT / "results" / "srm_composite_optimization_log.csv"
if srm_log_path.exists():
    srm_log = pd.read_csv(srm_log_path)
    srm_log["mean_validation_annual_dz"] = pd.to_numeric(srm_log["mean_validation_annual_dz"], errors="coerce")
    srm_log["annual_interval_gap"] = pd.to_numeric(srm_log["annual_interval_gap"], errors="coerce")
    srm_best = (
        srm_log.dropna(subset=["mean_validation_annual_dz"])
        .sort_values(["mean_validation_annual_dz", "annual_interval_gap"], ascending=[False, True])
        .iloc[0]
    )
else:
    print(f"SRM optimization log not found: {srm_log_path}")
    srm_best = pd.Series(dtype=object)

adaptive_best = final_summary.iloc[0]
comparison_rows = []
if not srm_best.empty:
    comparison_rows.append({
        "model": srm_best.get("model", "SRM Global Linear"),
        "role": "Current best model",
        "modulators": "none",
        "regularization": srm_best.get("param_regularization", "ridge/covariance shrinkage"),
        "z_clip": srm_best.get("param_z_clip", np.nan),
        "mean_annual_d_z": srm_best.get("mean_validation_annual_dz", np.nan),
        "dz_v1_v2": srm_best.get("dz_v1_v2", np.nan),
        "dz_v2_v3": srm_best.get("dz_v2_v3", np.nan),
        "annual_interval_gap": srm_best.get("annual_interval_gap", np.nan),
        "p_progression": srm_best.get("p_progression", np.nan),
        "pooled_pair_d_z_reference": srm_best.get("d_score", np.nan),
        "n_subject_pairs": srm_best.get("n_subjects", np.nan),
        "cv_mode": srm_best.get("cv_mode", np.nan),
        "source": "srm_composite_optimization_log.csv",
    })
comparison_rows.append({
    "model": "Patient-Adaptive",
    "role": "Adaptive interaction model",
    "modulators": adaptive_best.get("best_modulators", ", ".join(demographic_modulators)),
    "regularization": "ElasticNet interaction weights",
    "z_clip": adaptive_best.get("best_z_clip", np.nan),
    "mean_annual_d_z": adaptive_best.get("mean_annual_d_z", np.nan),
    "dz_v1_v2": adaptive_best.get("dz_v1_v2", np.nan),
    "dz_v2_v3": adaptive_best.get("dz_v2_v3", np.nan),
    "annual_interval_gap": adaptive_best.get("annual_interval_gap", np.nan),
    "p_progression": adaptive_best.get("p_progression", np.nan),
    "pooled_pair_d_z_reference": adaptive_best.get("pooled_pair_d_z_reference", np.nan),
    "n_subject_pairs": adaptive_best.get("n_subject_pairs", np.nan),
    "cv_mode": adaptive_best.get("cv_mode", np.nan),
    "source": "interaction_term.ipynb",
})

best_model_comparison = pd.DataFrame(comparison_rows)
metric_cols = ["mean_annual_d_z", "dz_v1_v2", "dz_v2_v3", "annual_interval_gap", "p_progression", "pooled_pair_d_z_reference", "n_subject_pairs"]
for col in metric_cols:
    if col in best_model_comparison:
        best_model_comparison[col] = pd.to_numeric(best_model_comparison[col], errors="coerce")

print("Patient-Adaptive vs current best model")
display(best_model_comparison)

if len(best_model_comparison) >= 2:
    srm_value = best_model_comparison.loc[best_model_comparison["role"].eq("Current best model"), "mean_annual_d_z"]
    adaptive_value = best_model_comparison.loc[best_model_comparison["model"].eq("Patient-Adaptive"), "mean_annual_d_z"]
    if len(srm_value) and len(adaptive_value):
        delta = float(adaptive_value.iloc[0] - srm_value.iloc[0])
        print(f"Mean annual d_z difference, Patient-Adaptive minus current best model: {delta:.3f}")


Patient-Adaptive vs current best model


,model,role,modulators,regularization,z_clip,mean_annual_d_z,dz_v1_v2,dz_v2_v3,annual_interval_gap,p_progression,pooled_pair_d_z_reference,n_subject_pairs,cv_mode,source
0,SRM Global Linear nested,Current best model,none,NaN,NaN,0.954528,1.129187,0.779868,0.349319,0.823232,0.935054,207,group_kfold,srm_composite_optimization_log.csv
1,Patient-Adaptive,Adaptive interaction model,"age, sex, gaa_1, gaa_2, onset_age, disease_dur...",ElasticNet interaction weights,4.0,0.559325,0.679098,0.439552,0.239545,0.718160,0.552491,202,group_kfold,interaction_term.ipynb


Mean annual d_z difference, Patient-Adaptive minus current best model: -0.395


## 5. Save Patient-Adaptive Results

This notebook writes the Patient-Adaptive optimization log and appends a model-performance-compatible CSV for downstream reporting.


In [12]:
interaction_optimization_log = optimization_log(optimization_rows, sort_by="mean_validation_annual_dz")
optimization_log_path = save_optimization_log(interaction_optimization_log, REPO_ROOT / "results" / "interaction_term_optimization_log.csv")
print(f"Saved Patient-Adaptive optimization log: {optimization_log_path}")

composite_intervals = adaptive_intervals.rename(columns={
    "d_z_ci_low": "d_z_ci_low",
    "d_z_ci_high": "d_z_ci_high",
})
performance_rows = assemble_performance_rows(
    "Patient-Adaptive",
    composite_intervals=composite_intervals,
    cv_mode=f"subject-level grouped {ADAPTIVE_CV_N_SPLITS}-fold",
    source="interaction_term.ipynb",
)
performance_csv = save_one_performance_csv(performance_rows, REPO_ROOT / "results" / "patient_adaptive_performance_summary.csv")
print(f"Saved Patient-Adaptive performance CSV: {performance_csv}")
display(performance_rows)


Saved Patient-Adaptive optimization log: /Users/robertwang/Documents/New_project/biomarkers/results/interaction_term_optimization_log.csv
Saved Patient-Adaptive performance CSV: /Users/robertwang/Documents/New_project/biomarkers/results/patient_adaptive_performance_summary.csv


,model,question,metric,role,value,n,status,evidence,cv_mode,source
0,Patient-Adaptive,12-month sensitivity V1->V2,"V1->V2 paired d_z, CI, N, P(delta>0)",Primary,"0.6790975158916651 [0.49656716730743106, 0.880...",106.0,computed,composite V1->V2 OOF annual interval,subject-level grouped 5-fold,interaction_term.ipynb
1,Patient-Adaptive,12-month sensitivity V2->V3,"V2->V3 paired d_z, CI, N, P(delta>0)",Primary temporal replication,"0.4395520223983853 [0.2573199558626743, 0.6570...",96.0,computed,composite V2->V3 OOF annual interval,subject-level grouped 5-fold,interaction_term.ipynb
2,Patient-Adaptive,12-month pooled annual sensitivity,"Pooled V1->V2 + V2->V3 paired d_z, CI, N, P(de...",Pooled annual diagnostic,NaN,NaN,missing,pooled annual V1->V2 + V2->V3,subject-level grouped 5-fold,interaction_term.ipynb
3,Patient-Adaptive,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,NaN,NaN,missing,composite V1->V3 cumulative,subject-level grouped 5-fold,interaction_term.ipynb
4,Patient-Adaptive,Direction consistency,P(delta > 0),Secondary,0.8113207547169812; 0.625,106.0,computed,annual V1->V2 and V2->V3 P(delta>0),subject-level grouped 5-fold,interaction_term.ipynb
5,Patient-Adaptive,Robustness,bootstrap CI for d_z,Primary uncertainty,"V1->V2 [0.49656716730743106, 0.880979669860738...",106.0,computed,annual interval bootstrap CI,subject-level grouped 5-fold,interaction_term.ipynb
6,Patient-Adaptive,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,nan vs nan,NaN,missing_reference,clinical interval benchmark,subject-level grouped 5-fold,interaction_term.ipynb
7,Patient-Adaptive,Better than MRI alone?,vs strongest individual MRI feature,RQ1,nan vs nan: nan,NaN,missing_reference,strongest single MRI feature,subject-level grouped 5-fold,interaction_term.ipynb
8,Patient-Adaptive,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,missing,FRDA vs control change,subject-level grouped 5-fold,interaction_term.ipynb
9,Patient-Adaptive,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN,NaN,missing,cross-sectional Spearman,subject-level grouped 5-fold,interaction_term.ipynb
